In [0]:
import requests                # conversa com a API (HTTP)

url = "https://brasilapi.com.br/api/feriados/v1/2026"
resposta = requests.get(url, timeout=30)


feriados = resposta.json()     # o JSON vira uma lista do Python: um dicionário por feriado
print(feriados)


[{'date': '2026-01-01', 'name': 'Confraternização mundial', 'type': 'national', 'weekday': 'quinta-feira'}, {'date': '2026-02-16', 'name': 'Carnaval', 'type': 'national', 'weekday': 'segunda-feira'}, {'date': '2026-02-17', 'name': 'Carnaval', 'type': 'national', 'weekday': 'terça-feira'}, {'date': '2026-04-03', 'name': 'Sexta-feira Santa', 'type': 'national', 'weekday': 'sexta-feira'}, {'date': '2026-04-05', 'name': 'Páscoa', 'type': 'national', 'weekday': 'domingo'}, {'date': '2026-04-21', 'name': 'Tiradentes', 'type': 'national', 'weekday': 'terça-feira'}, {'date': '2026-05-01', 'name': 'Dia do trabalho', 'type': 'national', 'weekday': 'sexta-feira'}, {'date': '2026-06-04', 'name': 'Corpus Christi', 'type': 'national', 'weekday': 'quinta-feira'}, {'date': '2026-09-07', 'name': 'Independência do Brasil', 'type': 'national', 'weekday': 'segunda-feira'}, {'date': '2026-10-12', 'name': 'Nossa Senhora Aparecida', 'type': 'national', 'weekday': 'segunda-feira'}, {'date': '2026-11-02', 'nam

In [0]:
import json, os
from datetime import datetime

pasta = "/Volumes/lakehouse_vertice/bronze/raw/feriados"
os.makedirs(pasta, exist_ok=True)                        # a pasta nova precisa existir

agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")     # ex.: 2026-10-20_19-40-00 (horário UTC)
with open(f"{pasta}/feriados_{agora}.json", "w") as arquivo:
    json.dump(feriados, arquivo, ensure_ascii=False)

sorted(os.listdir(pasta))                                # um arquivo por execução


['feriados_2026-10-21_16-58-27.json']

In [0]:
%sql
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.feriados
COMMENT 'Feriados nacionais de 2026, da BrasilAPI. Uma captura por execução do job.'
AS
SELECT
    date AS data,
    name AS feriado,
    type AS tipo,
    weekday AS dia_da_semana,
    _metadata.file_name AS arquivo,
    _metadata.file_modification_time AS coletado_em
FROM read_files(
    '/Volumes/lakehouse_vertice/bronze/raw/feriados/',
    format => 'json',
    multiline => true
)

num_affected_rows,num_inserted_rows
